In [21]:
import numpy as np
import pandas as pd
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer

In [2]:
train = pd.read_csv("Train.csv", index_col="PassengerId")
test = pd.read_csv("Test.csv", index_col="PassengerId")

In [3]:
train.columns

Index(['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'Transported',
       'GroupSize', 'Deck', 'Num', 'Side', 'Expenses'],
      dtype='str')

In [4]:
train.isna().sum().sort_values(ascending=False)

CryoSleep      217
VIP            203
HomePlanet     201
Deck           199
Num            199
Side           199
Destination    182
Age            179
Transported      0
GroupSize        0
Expenses         0
dtype: int64

In [8]:
train["CryoSleep"] = (train["CryoSleep"] == "True").astype(int)
test["CryoSleep"] = (test["CryoSleep"] == "True").astype(int)

train["VIP"] = (train["VIP"] == "True").astype(int)
test["VIP"] = (test["VIP"] == "True").astype(int)

train["Side"] = (train["Side"] == "S").astype(int)
test["CryoSleep"] = (test["CryoSleep"] == "S").astype(int)

In [ ]:
#Jetzt Null Werte bearbeiten

In [16]:
categorical_cols = ["Deck", "HomePlanet", "Side", "Destination"]

In [17]:
categorical_cols

['Deck', 'HomePlanet', 'Side', 'Destination']

In [13]:
#Cryosleep imputer
class GroupedModeImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Modus pro Deck-Gruppe merken (nur aus Trainingsdaten!)
        self.group_modes_ = (
            X.groupby(self.group_col)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
        )
        # Fallback für Decks, die im Train nicht vorkamen
        self.global_mode_ = X[self.target_col].mode().iloc[0]
        return self

    def transform(self, X):
        X = X.copy()
        missing = X[self.target_col].isna()
        fill_values = X.loc[missing, self.group_col].map(self.group_modes_)
        fill_values = fill_values.fillna(self.global_mode_)  # falls Deck unbekannt oder Deck-Modus selbst NaN
        X.loc[missing, self.target_col] = fill_values
        return X

In [23]:
cat_cols = ["Deck", "HomePlanet", "Side", "Destination"]

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

ct = ColumnTransformer([
    ("cat", cat_pipe, cat_cols),
    ("age_impute", SimpleImputer(strategy="median"), ["Age", "Num"]),
], remainder="passthrough")

In [24]:
X = train.drop(columns=["Transported"])
y = train["Transported"]

In [25]:
Baseline = Pipeline([
    ("cryo_impute", GroupedModeImputer(group_col="Deck", target_col="CryoSleep")),
    ("preprocess", ct),
    ("model", LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 20),
        cv=5,
        scoring="accuracy",
        max_iter=1000,
    )),
])

In [26]:
cross_val_score(Baseline, X, y, cv=5, scoring="accuracy")

/Users/andyzhu/PycharmProjects/Kaggle_Spaceship_Titanic/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/andyzhu/PycharmProjects/Kaggle_Spaceship_Titanic/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegress

array([0.67855089, 0.68832662, 0.69292697, 0.6990794 , 0.66340621])